# NL → SQL com DSPy + Bot Telegram
**Tema: Supermercado** — Categorias de produtos e suas características

**Melhorias aplicadas:**
- Signatures com `desc` detalhado e regras de negócio
- Otimizador `BootstrapFewShot` (few-shots automáticos)
- Resposta em linguagem natural via terceira Signature
- Integração com bot Telegram via `python-telegram-bot`

**Schema do banco:**
- `categorias` — seções do supermercado (ex: Laticínios, Hortifruti)
- `produtos` — itens com preço, peso, marca, validade e estoque
- `caracteristicas` — atributos extras por produto (ex: vegano, sem glúten, orgânico)

## 1. Instalação de dependências

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!pip install dspy-ai python-telegram-bot --quiet

In [ ]:
import threading, subprocess

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [ ]:
!ollama pull gemma3:1b

## 2. Imports e configuração do DSPy

In [ ]:
import dspy
import sqlite3
import requests

def setup_ollama():
    try:
        requests.get("http://localhost:11434/api/tags")
        print("✓ Ollama rodando")
    except:
        print("✗ Ollama não está rodando. Execute: ollama serve")
        return False

    dspy.settings.configure(
        lm=dspy.LM(
            model="ollama/gemma3:1b",
            api_base="http://localhost:11434",
            max_tokens=500,
            temperature=0.3,
        )
    )
    print("✓ DSPy configurado")

setup_ollama()

## 3. Banco de dados — Supermercado

Três tabelas relacionadas:
- **`categorias`** — seções do supermercado
- **`produtos`** — cadastro completo de cada item
- **`caracteristicas`** — atributos booleanos e textuais por produto (vegano, orgânico, etc.)

In [ ]:
def create_sample_database():
    conn = sqlite3.connect("supermercado.db")
    c = conn.cursor()

    # ── Tabelas ───────────────────────────────────────────────────────────────
    c.execute("""
        CREATE TABLE IF NOT EXISTS categorias (
            id          INTEGER PRIMARY KEY,
            nome        TEXT NOT NULL,
            corredor    INTEGER NOT NULL,
            responsavel TEXT NOT NULL
        )
    """)

    c.execute("""
        CREATE TABLE IF NOT EXISTS produtos (
            id            INTEGER PRIMARY KEY,
            nome          TEXT NOT NULL,
            categoria_id  INTEGER NOT NULL REFERENCES categorias(id),
            marca         TEXT NOT NULL,
            preco         REAL NOT NULL,
            peso_kg       REAL NOT NULL,
            estoque       INTEGER NOT NULL,
            validade      TEXT NOT NULL
        )
    """)

    c.execute("""
        CREATE TABLE IF NOT EXISTS caracteristicas (
            id           INTEGER PRIMARY KEY,
            produto_id   INTEGER NOT NULL REFERENCES produtos(id),
            atributo     TEXT NOT NULL,
            valor        TEXT NOT NULL
        )
    """)

    # ── Categorias ────────────────────────────────────────────────────────────
    c.executemany("INSERT OR IGNORE INTO categorias VALUES (?, ?, ?, ?)", [
        (1, 'Laticínios',   1, 'Fernanda Lima'),
        (2, 'Hortifruti',   2, 'Carlos Mendes'),
        (3, 'Padaria',      3, 'Ana Souza'),
        (4, 'Bebidas',      4, 'Ricardo Alves'),
        (5, 'Limpeza',      5, 'Patrícia Costa'),
        (6, 'Congelados',   6, 'Marcos Oliveira'),
        (7, 'Mercearia',    7, 'Juliana Ferreira'),
    ])

    # ── Produtos ──────────────────────────────────────────────────────────────
    c.executemany("INSERT OR IGNORE INTO produtos VALUES (?, ?, ?, ?, ?, ?, ?, ?)", [
        # Laticínios
        (1,  'Leite Integral 1L',        1, 'Itambé',       4.99,  1.00, 120, '2025-08-01'),
        (2,  'Iogurte Natural 170g',      1, 'Danone',       3.49,  0.17,  80, '2025-07-15'),
        (3,  'Queijo Mussarela 500g',     1, 'Polenghi',    18.90,  0.50,  45, '2025-07-20'),
        (4,  'Manteiga com Sal 200g',     1, 'Aviação',      9.80,  0.20,  60, '2025-09-10'),
        # Hortifruti
        (5,  'Banana Prata kg',           2, 'Granel',       5.90,  1.00, 200, '2025-06-25'),
        (6,  'Tomate Italiano kg',        2, 'Granel',       7.50,  1.00, 150, '2025-06-22'),
        (7,  'Alface Crespa un',          2, 'Verde Campo',  2.99,  0.30,  90, '2025-06-20'),
        (8,  'Maçã Gala kg',              2, 'Granel',       8.90,  1.00, 130, '2025-07-05'),
        # Padaria
        (9,  'Pão Francês 50g',           3, 'Própria',      0.79,  0.05, 500, '2025-06-14'),
        (10, 'Bolo de Cenoura 400g',      3, 'Própria',     12.50,  0.40,  20, '2025-06-16'),
        (11, 'Pão de Forma Integral',     3, 'Wickbold',     9.90,  0.50,  55, '2025-07-01'),
        # Bebidas
        (12, 'Água Mineral 1,5L',         4, 'Crystal',      2.49,  1.50, 300, '2026-12-31'),
        (13, 'Suco de Laranja 1L',        4, 'Del Valle',    7.99,  1.00,  70, '2025-09-30'),
        (14, 'Refrigerante Cola 2L',      4, 'Coca-Cola',    9.50,  2.00,  85, '2025-11-15'),
        (15, 'Cerveja Lager 350ml',       4, 'Brahma',       4.20,  0.35, 240, '2025-10-01'),
        # Limpeza
        (16, 'Detergente 500ml',          5, 'Ypê',          2.99,  0.50, 180, '2027-01-01'),
        (17, 'Sabão em Pó 1kg',           5, 'OMO',         18.90,  1.00,  95, '2027-06-01'),
        (18, 'Desinfetante 2L',           5, 'Pinho Sol',    9.90,  2.00, 110, '2027-03-01'),
        # Congelados
        (19, 'Pizza Calabresa 460g',      6, 'Sadia',       22.90,  0.46,  40, '2025-12-01'),
        (20, 'Frango Inteiro Cong. kg',   6, 'Perdigão',    16.50,  1.00,  60, '2025-11-30'),
        (21, 'Sorvete Creme 2L',          6, 'Kibon',       24.90,  2.00,  35, '2026-01-15'),
        # Mercearia
        (22, 'Arroz Branco 5kg',          7, 'Tio João',    28.90,  5.00, 150, '2026-06-01'),
        (23, 'Feijão Carioca 1kg',        7, 'Camil',        8.90,  1.00, 130, '2026-04-01'),
        (24, 'Azeite Extravirgem 500ml',  7, 'Gallo',       42.90,  0.50,  50, '2026-08-01'),
        (25, 'Macarrão Espaguete 500g',   7, 'Barilla',      6.50,  0.50, 200, '2026-05-01'),
    ])

    # ── Características ───────────────────────────────────────────────────────
    c.executemany("INSERT OR IGNORE INTO caracteristicas VALUES (?, ?, ?, ?)", [
        # Laticínios
        (1,  1,  'sem_lactose',   'nao'),
        (2,  1,  'origem',        'animal'),
        (3,  2,  'sem_lactose',   'nao'),
        (4,  2,  'probiotico',    'sim'),
        (5,  3,  'sem_lactose',   'nao'),
        (6,  4,  'sem_lactose',   'nao'),
        # Hortifruti
        (7,  5,  'organico',      'nao'),
        (8,  5,  'vegano',        'sim'),
        (9,  6,  'organico',      'sim'),
        (10, 6,  'vegano',        'sim'),
        (11, 7,  'organico',      'sim'),
        (12, 7,  'vegano',        'sim'),
        (13, 8,  'organico',      'nao'),
        (14, 8,  'vegano',        'sim'),
        # Padaria
        (15, 9,  'sem_gluten',    'nao'),
        (16, 9,  'vegano',        'sim'),
        (17, 10, 'sem_gluten',    'nao'),
        (18, 10, 'vegano',        'nao'),
        (19, 11, 'sem_gluten',    'nao'),
        (20, 11, 'integral',      'sim'),
        # Bebidas
        (21, 12, 'com_gas',       'nao'),
        (22, 12, 'vegano',        'sim'),
        (23, 13, 'com_gas',       'nao'),
        (24, 13, 'organico',      'nao'),
        (25, 14, 'com_gas',       'sim'),
        (26, 14, 'vegano',        'sim'),
        (27, 15, 'com_gas',       'sim'),
        (28, 15, 'alcoolico',     'sim'),
        # Limpeza
        (29, 16, 'biodegradavel', 'sim'),
        (30, 17, 'biodegradavel', 'nao'),
        (31, 18, 'biodegradavel', 'sim'),
        # Congelados
        (32, 19, 'sem_gluten',    'nao'),
        (33, 19, 'vegano',        'nao'),
        (34, 20, 'vegano',        'nao'),
        (35, 21, 'vegano',        'nao'),
        # Mercearia
        (36, 22, 'organico',      'nao'),
        (37, 22, 'sem_gluten',    'sim'),
        (38, 23, 'organico',      'nao'),
        (39, 23, 'sem_gluten',    'sim'),
        (40, 24, 'organico',      'sim'),
        (41, 24, 'vegano',        'sim'),
        (42, 25, 'sem_gluten',    'nao'),
        (43, 25, 'integral',      'nao'),
    ])

    conn.commit()
    conn.close()
    print("✓ Banco supermercado.db criado com 7 categorias, 25 produtos e 43 características")

create_sample_database()

## 4. Signatures

O `SCHEMA_DESC` agora descreve o domínio do supermercado com regras específicas
para os JOINs e o campo `caracteristicas` (que usa atributo/valor como EAV).

In [ ]:
SCHEMA_DESC = """
Schema do banco SQLite — Supermercado:

  categorias   : id (INT PK), nome (TEXT), corredor (INT), responsavel (TEXT)
  produtos     : id (INT PK), nome (TEXT), categoria_id (INT FK→categorias.id),
                 marca (TEXT), preco (REAL), peso_kg (REAL), estoque (INT),
                 validade (TEXT 'YYYY-MM-DD')
  caracteristicas: id (INT PK), produto_id (INT FK→produtos.id),
                   atributo (TEXT), valor (TEXT)

Valores possíveis de atributo em caracteristicas:
  'vegano', 'organico', 'sem_gluten', 'sem_lactose', 'integral',
  'probiotico', 'com_gas', 'alcoolico', 'biodegradavel', 'origem'

Valores possíveis de valor em caracteristicas: 'sim', 'nao' (ou texto livre para 'origem')

Categorias existentes (nome exato): Laticínios, Hortifruti, Padaria, Bebidas, Limpeza, Congelados, Mercearia

Regras obrigatórias:
  1. Gere apenas SELECT — nunca INSERT, UPDATE, DELETE ou DDL.
  2. Use nomes de tabela e coluna EXATAMENTE como no schema acima (acentos incluídos).
  3. Para buscar produtos por categoria: JOIN produtos p ON p.categoria_id = c.id
  4. Para buscar produtos por característica: JOIN caracteristicas cr ON cr.produto_id = p.id
     WHERE cr.atributo = '<atributo>' AND cr.valor = '<valor>'
  5. Para filtrar validade use: validade < 'YYYY-MM-DD' (vencendo antes) ou > (após).
  6. Retorne apenas o SQL, sem markdown, sem explicação.
"""


# ─── Signature 1: Linguagem natural → SQL ────────────────────────────────────
class GenerateSQL(dspy.Signature):
    """Converta uma pergunta sobre o supermercado em uma query SQL válida."""

    schema_info: str = dspy.InputField(
        desc="Schema completo do banco de dados do supermercado e regras de geração SQL"
    )
    question: str = dspy.InputField(
        desc="Pergunta do usuário em linguagem natural (português ou inglês) sobre produtos, categorias ou características"
    )
    sql_query: str = dspy.OutputField(
        desc="Query SQL válida, somente SELECT, sem markdown nem explicação adicional"
    )


# ─── Signature 2: SQL com erro → SQL corrigido ────────────────────────────────
class RefineSQL(dspy.Signature):
    """Corrija uma query SQL do supermercado que retornou um erro de execução."""

    schema_info: str = dspy.InputField(
        desc="Schema completo do banco de dados do supermercado e regras de geração SQL"
    )
    question: str = dspy.InputField(
        desc="Pergunta original do usuário"
    )
    sql_query: str = dspy.InputField(
        desc="Query SQL que falhou"
    )
    error: str = dspy.InputField(
        desc="Mensagem de erro retornada pelo SQLite"
    )
    refined_sql: str = dspy.OutputField(
        desc="Query SQL corrigida, sem markdown nem explicação adicional"
    )


# ─── Signature 3: Resultado bruto → Resposta em linguagem natural ─────────────
class NaturalAnswer(dspy.Signature):
    """Transforme o resultado de uma consulta ao supermercado em resposta amigável em português."""

    question: str = dspy.InputField(
        desc="Pergunta original feita pelo usuário sobre o supermercado"
    )
    sql_query: str = dspy.InputField(
        desc="Query SQL que foi executada"
    )
    raw_results: str = dspy.InputField(
        desc="Resultado bruto retornado pelo banco, como lista de tuplas Python"
    )
    answer: str = dspy.OutputField(
        desc=(
            "Resposta clara e direta em português brasileiro sobre os produtos ou categorias. "
            "Inclua os dados relevantes (nomes, preços, quantidades). "
            "Exemplos: 'Há 4 produtos na categoria Laticínios.' / "
            "'O produto mais caro é o Azeite Extravirgem Gallo por R$ 42,90.' "
            "Se não houver resultados, informe que nenhum produto foi encontrado."
        )
    )

print("✓ Signatures definidas")

## 5. Módulo SQLGenerator

In [ ]:
class SQLGenerator(dspy.Module):
    """Gera SQL, executa, refina em caso de erro e responde em linguagem natural."""

    def __init__(self):
        super().__init__()
        self.generator = dspy.ChainOfThought(GenerateSQL)
        self.refiner   = dspy.ChainOfThought(RefineSQL)
        self.answerer  = dspy.ChainOfThought(NaturalAnswer)

    def forward(self, question: str, conn: sqlite3.Connection) -> dspy.Prediction:
        # ── Passo 1: gerar SQL ────────────────────────────────────────────────
        output = self.generator(schema_info=SCHEMA_DESC, question=question)
        sql = output.sql_query.strip()

        # ── Passo 2: executar (e refinar se falhar) ───────────────────────────
        try:
            results = conn.execute(sql).fetchall()
        except Exception as e:
            refined = self.refiner(
                schema_info=SCHEMA_DESC,
                question=question,
                sql_query=sql,
                error=str(e),
            )
            sql = refined.refined_sql.strip()
            try:
                results = conn.execute(sql).fetchall()
            except Exception as e2:
                return dspy.Prediction(
                    sql_query=sql, results=None, error=str(e2),
                    answer=f"Não foi possível executar a consulta. Erro: {e2}",
                )

        # ── Passo 3: resposta em linguagem natural ────────────────────────────
        natural = self.answerer(
            question=question,
            sql_query=sql,
            raw_results=str(results),
        )

        return dspy.Prediction(
            sql_query=sql,
            results=results,
            answer=natural.answer,
            error=None,
        )

print("✓ Módulo SQLGenerator definido")

## 6. Otimização com BootstrapFewShot

O trainset cobre os tipos de consulta mais comuns no domínio do supermercado:
filtros por categoria, por preço, por estoque, por característica e por validade.

In [ ]:
from dspy.teleprompt import BootstrapFewShot

trainset = [
    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Quantos produtos existem na categoria Laticínios?",
        sql_query="SELECT COUNT(*) FROM produtos p JOIN categorias c ON p.categoria_id = c.id WHERE c.nome = 'Laticínios'"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Qual o produto mais caro do supermercado?",
        sql_query="SELECT nome, preco, marca FROM produtos ORDER BY preco DESC LIMIT 1"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Liste todos os produtos veganos disponíveis",
        sql_query="SELECT p.nome, p.preco, p.marca FROM produtos p JOIN caracteristicas cr ON cr.produto_id = p.id WHERE cr.atributo = 'vegano' AND cr.valor = 'sim'"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Quais produtos orgânicos estão no Hortifruti?",
        sql_query="SELECT p.nome, p.preco FROM produtos p JOIN categorias c ON p.categoria_id = c.id JOIN caracteristicas cr ON cr.produto_id = p.id WHERE c.nome = 'Hortifruti' AND cr.atributo = 'organico' AND cr.valor = 'sim'"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Mostre os produtos com estoque abaixo de 50 unidades",
        sql_query="SELECT nome, estoque, marca FROM produtos WHERE estoque < 50 ORDER BY estoque ASC"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Qual a média de preço dos produtos de Bebidas?",
        sql_query="SELECT AVG(p.preco) FROM produtos p JOIN categorias c ON p.categoria_id = c.id WHERE c.nome = 'Bebidas'"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Quais produtos vencem antes de 2025-07-01?",
        sql_query="SELECT nome, validade, marca FROM produtos WHERE validade < '2025-07-01' ORDER BY validade ASC"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Liste os produtos sem glúten com seus preços",
        sql_query="SELECT p.nome, p.preco, p.marca FROM produtos p JOIN caracteristicas cr ON cr.produto_id = p.id WHERE cr.atributo = 'sem_gluten' AND cr.valor = 'sim'"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Qual categoria tem mais produtos cadastrados?",
        sql_query="SELECT c.nome, COUNT(p.id) as total FROM categorias c JOIN produtos p ON p.categoria_id = c.id GROUP BY c.id ORDER BY total DESC LIMIT 1"
    ).with_inputs("schema_info", "question"),

    dspy.Example(
        schema_info=SCHEMA_DESC,
        question="Mostre todos os produtos do corredor 1",
        sql_query="SELECT p.nome, p.preco, p.marca FROM produtos p JOIN categorias c ON p.categoria_id = c.id WHERE c.corredor = 1"
    ).with_inputs("schema_info", "question"),
]

print(f"✓ Trainset criado com {len(trainset)} exemplos")

In [ ]:
_metric_conn = sqlite3.connect("supermercado.db")

def sql_executable_metric(example, prediction, trace=None):
    """Retorna True se o SQL gerado executar sem erro no banco."""
    sql = getattr(prediction, "sql_query", "").strip()
    if not sql:
        return False
    try:
        _metric_conn.execute(sql).fetchall()
        return True
    except Exception:
        return False

print("✓ Métrica definida")

In [ ]:
optimizer = BootstrapFewShot(
    metric=sql_executable_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
)

class GeneratorForOptimizer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generator = dspy.ChainOfThought(GenerateSQL)

    def forward(self, schema_info, question):
        return self.generator(schema_info=schema_info, question=question)

base_generator = SQLGenerator()

optimized_gen = optimizer.compile(
    GeneratorForOptimizer(),
    trainset=trainset,
)

base_generator.generator = optimized_gen.generator
generator = base_generator

print("✓ Otimização concluída — few-shots automáticos aplicados")

## 7. Testes rápidos

In [ ]:
conn = sqlite3.connect("supermercado.db")

perguntas_teste = [
    "Quantos produtos existem na categoria Laticínios?",
    "Qual o produto mais caro do supermercado?",
    "Liste todos os produtos veganos disponíveis",
    "Quais produtos vencem antes de 2025-07-01?",
    "Mostre os produtos com estoque abaixo de 50 unidades",
    "Quais bebidas alcoólicas estão disponíveis?",
    "Qual categoria tem o maior número de produtos?",
]

for i, pergunta in enumerate(perguntas_teste, 1):
    print(f"\n{'='*65}")
    print(f"Pergunta {i}: {pergunta}")
    result = generator(pergunta, conn)
    print(f"SQL      : {result.sql_query}")
    if result.error:
        print(f"Erro     : {result.error}")
    else:
        print(f"Resultado: {result.results}")
    print(f"Resposta : {result.answer}")

## 8. Bot Telegram

### Pré-requisitos
1. Crie um bot no Telegram via [@BotFather](https://t.me/BotFather) e copie o token.
2. Cole o token na célula abaixo.
3. Execute a célula — o bot ficará ouvindo mensagens.

### Comandos disponíveis
- Mensagem livre → resposta em linguagem natural sobre os produtos
- `/sql <pergunta>` → resposta + SQL gerado (útil para debug)
- `/categorias` → lista todas as categorias disponíveis
- `/start` e `/help` → instruções de uso

In [ ]:
TELEGRAM_TOKEN = "8893699149:AAGRrrHTiYCQT0TZ0aapcxIICXAABLaDqGQ"

In [ ]:
from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters,
)

DB_PATH = "supermercado.db"


async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "🛒 Olá! Sou o assistente do supermercado.\n\n"
        "Faça perguntas sobre produtos, preços, estoque e características.\n\n"
        "Exemplos:\n"
        "  • Quais produtos veganos estão disponíveis?\n"
        "  • Qual o produto mais barato da Padaria?\n"
        "  • Mostre produtos com estoque abaixo de 50\n"
        "  • Quais itens vencem essa semana?\n"
        "  • Tem algum produto orgânico no Hortifruti?\n\n"
        "Use /sql <pergunta> para ver também a query SQL gerada.\n"
        "Use /categorias para listar as seções disponíveis."
    )


async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "📖 *Como usar:*\n\n"
        "Envie qualquer pergunta sobre o supermercado.\n"
        "Use /sql <pergunta> para ver o SQL gerado.\n"
        "Use /categorias para ver as seções disponíveis.",
        parse_mode="Markdown",
    )


async def categorias_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Lista todas as categorias diretamente do banco, sem passar pelo LLM."""
    conn = sqlite3.connect(DB_PATH)
    try:
        rows = conn.execute(
            "SELECT nome, corredor, responsavel FROM categorias ORDER BY corredor"
        ).fetchall()
        texto = "🗂 *Categorias disponíveis:*\n\n"
        for nome, corredor, responsavel in rows:
            texto += f"  Corredor {corredor} — *{nome}* (resp: {responsavel})\n"
        await update.message.reply_text(texto, parse_mode="Markdown")
    finally:
        conn.close()


async def handle_question(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handler para mensagens livres — retorna resposta em linguagem natural."""
    question = update.message.text.strip()
    await update.message.reply_text("⏳ Consultando o estoque...")

    conn = sqlite3.connect(DB_PATH)
    try:
        result = generator(question, conn)
        await update.message.reply_text(result.answer)
    except Exception as e:
        await update.message.reply_text(f"❌ Erro inesperado: {e}")
    finally:
        conn.close()


async def handle_sql_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Handler para /sql <pergunta> — retorna resposta + SQL formatado."""
    if not context.args:
        await update.message.reply_text("Uso: /sql <sua pergunta>")
        return

    question = " ".join(context.args)
    await update.message.reply_text("⏳ Consultando o estoque...")

    conn = sqlite3.connect(DB_PATH)
    try:
        result = generator(question, conn)
        resposta = (
            f"{result.answer}\n\n"
            f"```sql\n{result.sql_query}\n```"
        )
        await update.message.reply_text(resposta, parse_mode="Markdown")
    except Exception as e:
        await update.message.reply_text(f"❌ Erro inesperado: {e}")
    finally:
        conn.close()


# ── Iniciar o bot ─────────────────────────────────────────────────────────────
app = (
    ApplicationBuilder()
    .token(TELEGRAM_TOKEN)
    .build()
)

app.add_handler(CommandHandler("start",      start))
app.add_handler(CommandHandler("help",       help_command))
app.add_handler(CommandHandler("categorias", categorias_command))
app.add_handler(CommandHandler("sql",        handle_sql_command))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_question))

print("🤖 Bot iniciado. Pressione Ctrl+C para parar.")
app.run_polling()

---
## Próximos passos sugeridos

| Melhoria | Como fazer |
|---|---|
| Ampliar o trainset | Adicionar mais exemplos com JOINs triplos (produto + categoria + característica) |
| Métricas mais rígidas | Comparar resultado da query com valor esperado, não só executabilidade |
| Persistir o modelo otimizado | `generator.save('generator_supermercado.json')` para não reotimizar toda sessão |
| Banco real | Trocar `sqlite3` por `psycopg2` (PostgreSQL) — o restante do código não muda |
| Comando /produto no Telegram | Busca direta por nome sem passar pelo LLM |
| Alertas de estoque | Job periódico que notifica o responsável da categoria quando estoque < threshold |
